# 案例 01 · 榜单任务并发导致 CPU 打满，Feed 接口超时

> 沙箱实验：真实进程 + 阿里云 OpenAPI 模拟（响应带 `Mock: true`）。
> 云产品字段/命令与生产一致，资源规格与账单为模拟值。

## 场景

榜单预计算 cron 被改成每分钟执行且没有加锁，单轮任务未结束下一轮已启动，4 个忙循环进程把 8 核实例的 CPU 吃满，Feed P99 从 120ms 涨到 820ms，SLB 健康检查失败并摘除后端。

## 练习目标

- 用 `uptime` / `ps` 区分“资源不足”与“进程异常”
- 定位定时任务并发叠加的证据
- 先止血（kill）再改造（flock/nice/limit）
- 用 `cpu_guard.py` 完成自动化处置与校验

## 前置

- 仓库根目录执行 `python -m venv .venv && .venv/bin/pip install -r requirements.txt`
- 使用该 venv 的 Python kernel 打开本 Notebook；或运行 `tools/run_all_labs.py` 批量执行


In [ ]:
# 环境准备：定位仓库根目录并加载沙箱
import json, os, signal, ssl, socket, sys, time, gzip, shutil, urllib.parse, urllib.request
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "labs" / "common" / "xhs_lab.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("请在 xhs-backend-cloud-ops-lab 仓库内运行本 Notebook")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from labs.common import Lab

lab = Lab("case-01-cpu-runaway")
lab.reset(start_api=True)

lab.summary()


## 步骤 1 · 异常重现

下面注入一个可复现的真实故障，观察它对业务的影响。

In [ ]:
lab.inject_cpu_runaway(procs=4)
time.sleep(1)
print("已注入：4 个忙循环进程（模拟榜单任务并发）")


## 步骤 2 · 取证

按“入口 → 主机 → 进程 → 依赖 → 日志”的分层顺序收集证据，不要一上来就重启。

**系统负载**

In [ ]:
print(lab.sh("uptime"))

**CPU 排序进程**

In [ ]:
print(lab.sh("ps -eo pid,ppid,user,%cpu,etime,cmd --sort=-%cpu | head -8"))

**定时任务配置**

In [ ]:
print(lab.sh("cat /etc/cron.d/xhs-ranking"))

**业务探活**

In [ ]:
print(lab.curl(lab.api + "/api/v1/feed"))

**SLB 健康状态**

In [ ]:
print(lab.aliyun("slb DescribeHealthStatus --LoadBalancerId lb-bp1xhsslb01"))

## 步骤 3 · 判断

把证据收敛成一句结论，决定止血动作。

In [ ]:
fault = lab.state["faults"]["cpu"]
print("异常进程数:", len(fault["pids"]))
print("单进程 CPU:", fault["cpu"], "%（单核口径）")
print("cron 是否加锁:", "flock" in lab.sh("cat /etc/cron.d/xhs-ranking"))
print("判断：进程数随时间线性增加 → 定时任务并发叠加，而非流量突增")


## 步骤 4 · 手工处置（先止血）

执行最小必要的变更，优先恢复业务，再考虑根治。

In [ ]:
# 先 TERM/SIGKILL 清理任务进程（生产先用 TERM 保护事务）
for pid in lab.state["faults"]["cpu"]["pids"]:
    try:
        os.killpg(os.getpgid(pid), signal.SIGKILL)
    except OSError:
        pass

# 改造 crontab：串行 + 降优先级 + 限流
lab.write("/etc/cron.d/xhs-ranking",
          "*/5 * * * * xhs flock -n /var/lock/ranking.lock nice -n 10 "
          "python3 /srv/xhs/jobs/ranking_precompute.py --limit 2000\n")
lab.state["faults"].pop("cpu", None)
lab._save()
time.sleep(1)
print(lab.sh("ps -eo pid,%cpu,cmd --sort=-%cpu | head -5"))


## 步骤 5 · 验证

用可执行的断言确认恢复，而不是“看起来好了”。

In [ ]:
lab.check('异常进程已清除', bool('while True' not in lab.sh('ps aux')))
lab.check('cron 已加锁限流', bool('flock' in lab.path('/etc/cron.d/xhs-ranking').read_text()))
lab.check('服务健康', bool('HTTP_CODE=200' in lab.curl(lab.api + '/healthz')))

## 步骤 6 · 自动化沉淀

脚本采样 CPU 占用、识别业务组异常进程，记录现场后终止，并把 crontab 改写为 `flock + nice + --limit`，最后校验残留进程与业务探活。

下面重置沙箱、重新注入故障，然后直接调用生产可用的自动化脚本（`--apply`；缺省为 dry-run）。

In [ ]:
lab.reset(start_api=True)
lab.inject_cpu_runaway(procs=4)
time.sleep(1)
out = lab.auto('cpu_guard.py', '--apply')
print(out)

In [ ]:
lab.check('自动化处置返回 REMEDIATED', 'RESULT: REMEDIATED' in out)
print('\n自动化脚本已可作为 cron / 事件总线触发器复用')

## 复盘与思考题

1. 如果 CPU 高但流量正常、进程是 Java，你会如何进一步定位？
2. 为什么“CPU 高”不等于“要扩容”？
3. kill 后任务需要补跑，如何保证不再次叠加？

## 收尾

沙箱资源会在本 Cell 中停止；重新运行请从第 1 个代码 Cell 开始。


In [ ]:
lab.stop_all()
print('沙箱已停止')